# Analyse des données League of Legends

**Problématique** : Analyser et trouver quel est le meilleur champion et quels sont les rôles et les positions les plus représentés

#### Initialisation des librairies

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

path = "data/"

#### Initialisation des Datasets

##### Champion Dataset

In [ ]:
champs = pd.read_csv(path + "champs.csv")

##### Participants Dataset

In [ ]:
participants = pd.read_csv(path + "participants.csv")

##### Matches Dataset

In [ ]:
matches = pd.read_csv(path + "matches.csv")

##### Stat1 Dataset

In [ ]:
stats1 = pd.read_csv(path + "stats1.csv")

##### Stat2 Dataset

In [ ]:
stats2 = pd.read_csv(path + "stats2.csv", low_memory=False)

##### Team Bans Dataframe

In [ ]:
teambans = pd.read_csv(path + "teambans.csv")

##### Team Stats Dataframe

In [ ]:
teamstats = pd.read_csv(path + "teamstats.csv")

#### Tests et analyse

In [50]:
# python
# quick diagnostics
print(participants.shape)
print(participants.columns.tolist())
print(participants['position'].dtype, participants['role'].dtype)
print(participants['position'].isna().sum(), participants['role'].isna().sum())
print(participants[['position', 'role']].head())

# vectorized fix (fast)
mask_bot = participants['position'] == "BOT"
participants.loc[mask_bot & (participants['role'] == 'DUO_SUPPORT'), 'position'] = "SUPP"
participants.loc[mask_bot & (participants['role'] != 'DUO_SUPPORT'), 'position'] = "ADC"

# verify
participants.loc[:, ["position", "role"]].head()

print(participants['championid'].dtype, champs['id'].dtype)


(1834520, 9)
['id', 'matchid', 'player', 'championid', 'ss1', 'ss2', 'role', 'position', 'teamid']
str str
0 0
  position         role
0   JUNGLE         NONE
1     SUPP  DUO_SUPPORT
2      ADC    DUO_CARRY
3      TOP         SOLO
4      MID         SOLO
int64 int64


In [57]:
participants["teamid"] = participants.player.apply(lambda x: 100 if x <= 5 else 200)

keep_cols_stats = ["id", "win", "kills"]
keep_cols_participants = ['id', 'championid', 'player', 'ss1', 'ss2', 'position']

stats_merged = pd.concat([stats1[keep_cols_stats], stats2[keep_cols_stats]])
"""stats_merged = pd.merge(
    stats1,
    stats2,
    on='id',  # ou la clé commune appropriée
    how='inner',
    suffixes=('_s1', '_s2')
)"""

train_data = pd.merge(participants, stats_merged, on="id")
train_data = pd.merge(train_data, teamstats.drop(["firstblood"], axis=1), on=["matchid", "teamid"])

champs_info = champs[['id', 'name']].rename(columns={'name': 'champion_name'})

train_data = pd.merge(train_data, champs_info, left_on='championid', right_on='id', how='left', suffixes=('', '_champ'))

train_data = train_data.drop(
    ["id", "matchid", "teamid", "player", "firsttower", 'firstinhib', 'firstharry', 'firstbaron', 'firstdragon',
     'baronkills', 'dragonkills', 'harrykills', 'inhibkills', 'towerkills', 'id_champ', 'championid'], axis=1)
train_data[train_data.win == 1]

,ss1,ss2,role,position,win,kills,champion_name
5,11,4,NONE,JUNGLE,1,3,Skarner
6,4,12,SOLO,TOP,1,4,Galio
7,14,4,SOLO,MID,1,13,Ahri
8,7,4,DUO_CARRY,ADC,1,15,Jinx
9,14,4,DUO_SUPPORT,SUPP,1,4,VelKoz
...,...,...,...,...,...,...,...
1834507,4,12,SOLO,MID,1,3,Twisted Fate
1834508,7,4,DUO_CARRY,ADC,1,10,Tristana
1834509,4,11,NONE,JUNGLE,1,22,Xin Zhao
1834510,12,4,SOLO,TOP,1,7,Malphite
